# Early Stopping Analysis: BASELINE vs ES_PATIENCE
## Calpella Daily - Effect of Early Stopping on Hyperparameter Selection

**Research Question:** Does enabling early stopping during gridsearch affect hyperparameter selection and final model performance?

**Runs:**
- `BASELINE`: Fixed epochs (16/32/48), no early stopping
- `ES_PATIENCE`: Patience-based early stopping (patience=3, min_epochs=10)

In [1]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from UCB_training.UCB_eval import load_test_metrics, pairwise_pct_change, threshold_filter
from UCB_training.UCB_plotting import analyze_early_stopping_runs

OUTPUT_BASE = Path("../../outputs/calpella")
DAILY_METRICS = OUTPUT_BASE / "daily"
SHARED_RUNS = OUTPUT_BASE / "daily_shared" / "runs"

BASELINE_DIR = DAILY_METRICS / "BASELINE_20250815T000000Z"
ES_DIR = DAILY_METRICS / "ES_PATIENCE_20250815T000000Z"
ES_RUNS_DIR = SHARED_RUNS / "ES_PATIENCE"

print("Paths configured.")

Paths configured.


## 1. Early Stopping Behavior Analysis

First, analyze how often early stopping triggered during the gridsearch.

In [2]:
es_stats = analyze_early_stopping_runs(ES_RUNS_DIR)

print(f"Total runs: {es_stats['total_runs']}")
print(f"Early stopped: {es_stats['early_stopped_count']} ({es_stats['early_stopped_pct']:.1f}%)")
print()
print("By max_epochs configuration:")
print("-" * 60)
for max_ep, stats in es_stats['by_max_epochs'].items():
    print(f"max_epochs={max_ep}: {stats['runs']} runs, {stats['early_stopped']} early stopped ({stats['pct']:.1f}%)")
    if stats['stopped_at_epochs']:
        print(f"  -> stopped at epochs: {stats['stopped_at_epochs']}")

Total runs: 112
Early stopped: 44 (39.3%)

By max_epochs configuration:
------------------------------------------------------------
max_epochs=16: 40 runs, 0 early stopped (0.0%)
max_epochs=32: 36 runs, 12 early stopped (33.3%)
  -> stopped at epochs: [25, 30]
max_epochs=48: 36 runs, 32 early stopped (88.9%)
  -> stopped at epochs: [25, 30, 35, 40, 45]


### Early Stopping Summary

| max_epochs | runs | early stopped | rate | stopped at |
|------------|------|---------------|------|------------|
| 16 | 40 | 0 | 0% | (hit max before ES could trigger) |
| 32 | 36 | 12 | 33.3% | 25, 30 |
| 48 | 36 | 32 | 88.9% | 25-45 |

**Observation:** Early stopping is working. 39% of runs stopped early, with higher rates for longer training configurations.

## 2. Test Performance Comparison

In [3]:
baseline_metrics = load_test_metrics(BASELINE_DIR, "calpella")
es_metrics = load_test_metrics(ES_DIR, "calpella")

print("BASELINE metrics loaded:", baseline_metrics.shape)
print("ES_PATIENCE metrics loaded:", es_metrics.shape)

BASELINE metrics loaded: (14, 3)
ES_PATIENCE metrics loaded: (14, 3)


In [4]:
MODELS = ["LSTM", "PILSTM"]

comparison = pairwise_pct_change(baseline_metrics, es_metrics, models=MODELS)
comparison = comparison.rename(columns={"EXPERIMENTAL": "ES_PATIENCE"})
comparison.round(3)

,Metric,Model,BASELINE,ES_PATIENCE,pct_change
0,NSE,LSTM,0.802,0.808,0.756
1,NSE,PILSTM,0.841,0.829,1.418
2,MSE,LSTM,50422.880,48878.300,3.063
3,MSE,PILSTM,40461.058,43499.376,7.509
4,RMSE,LSTM,224.550,221.084,1.544
5,RMSE,PILSTM,201.149,208.565,3.687
6,KGE,LSTM,0.723,0.749,3.566
7,KGE,PILSTM,0.833,0.835,0.197
8,Alpha-NSE,LSTM,0.750,0.779,3.856
9,Alpha-NSE,PILSTM,0.876,0.862,1.537


## 3. Threshold Filtering

Filter to metrics with >5% change, always keeping NSE.

In [5]:
THRESHOLD = 5  # percent
ALWAYS_KEEP = ["NSE"]

# Need to add pct_change back for filtering
comparison_with_pct = pairwise_pct_change(baseline_metrics, es_metrics, models=MODELS)
filtered = threshold_filter(comparison_with_pct, threshold=THRESHOLD, always_keep=ALWAYS_KEEP)
filtered = filtered.rename(columns={"EXPERIMENTAL": "ES_PATIENCE"})

print(f"Kept {len(filtered)} of {len(comparison_with_pct)} rows (threshold={THRESHOLD}%, always keep: {ALWAYS_KEEP})")
filtered.round(3)

Kept 13 of 28 rows (threshold=5%, always keep: ['NSE'])


,Metric,Model,BASELINE,ES_PATIENCE
0,NSE,LSTM,0.802,0.808
1,NSE,PILSTM,0.841,0.829
3,MSE,PILSTM,40461.058,43499.376
11,Beta-KGE,PILSTM,0.924,0.977
13,Beta-NSE,PILSTM,-0.045,-0.013
16,FHV,LSTM,-23.218,-20.738
18,FMS,LSTM,-29.641,-18.674
19,FMS,PILSTM,-12.444,-23.350
20,FLV,LSTM,-78.330,-1011.230
21,FLV,PILSTM,54.654,71.940


## 4. Key Findings

*[To be filled after running cells]*